# Notebook 17 — Inert-Gas Compartment Models

**Companion to Chapter 17**

This laboratory propagates educational inert-gas compartment states. It intentionally does not compute ceilings, no-decompression limits, or schedules.

> **Never use this notebook for dive planning.** A mathematical model cannot guarantee freedom from decompression sickness.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})
rho, g, p0 = 1025.0, 9.80665, 101325.0

def ambient_pressure_pa(depth_m):
    return p0 + rho*g*np.asarray(depth_m)

def ambient_pressure_bar(depth_m):
    return ambient_pressure_pa(depth_m)/1e5

## 1. Inspired nitrogen pressure and surface equilibrium

In [ ]:
f_n2=0.79
p_h2o_bar=0.0627

def inspired_n2_bar(depth_m):
    return f_n2*(ambient_pressure_bar(depth_m)-p_h2o_bar)

p_surface=inspired_n2_bar(0.0)
print(f"Surface inspired N2 pressure: {p_surface:.4f} bar")

## 2. Verify the half-time rule

In [ ]:
def constant_response(t,p_initial,p_inspired,half_time):
    k=np.log(2)/half_time
    return p_inspired+(p_initial-p_inspired)*np.exp(-k*np.asarray(t))

half=20.0; target=2.25; initial=0.75
print(constant_response(half,initial,target,half))
assert np.isclose(constant_response(half,initial,target,half),(initial+target)/2)

## 3. Multiple time scales after a pressure step

In [ ]:
half_times=np.array([5.,20.,80.])
t=np.linspace(0,120,1201)
for half in half_times:
    plt.plot(t,constant_response(t,p_surface,inspired_n2_bar(20),half),label=f"{half:g} min")
plt.xlabel("Time [min]"); plt.ylabel("Compartment N2 tension [bar]")
plt.title("Fast and slow compartment responses"); plt.legend(); plt.show()

## 4. Exact piecewise-constant state update

In [ ]:
def example_profile(t_min):
    t=np.asarray(t_min)
    return np.piecewise(t,[t<3,(t>=3)&(t<23),(t>=23)&(t<26),t>=26],
        [lambda x:20*x/3,20,lambda x:20*(26-x)/3,0])

def simulate_compartments(t,depth,half_times):
    state=np.empty((len(half_times),len(t))); state[:,0]=p_surface
    k=np.log(2)/np.asarray(half_times)
    for j in range(len(t)-1):
        dt=t[j+1]-t[j]; pi=inspired_n2_bar(depth[j])
        state[:,j+1]=pi+(state[:,j]-pi)*np.exp(-k*dt)
    return state

t=np.linspace(0,180,18001); z=example_profile(t)
states=simulate_compartments(t,z,half_times)
fig,ax=plt.subplots(2,1,sharex=True,figsize=(8,7))
ax[0].plot(t,z); ax[0].invert_yaxis(); ax[0].set_ylabel("Depth [m]")
for i,h in enumerate(half_times): ax[1].plot(t,states[i],label=f"{h:g} min")
ax[1].set(xlabel="Time [min]",ylabel="N2 tension [bar]"); ax[1].legend(); plt.show()

## 5. Equal current depth, different histories

In [ ]:
t2=np.linspace(0,30,3001)
profile_a=np.where(t2<3,20*t2/3,np.where(t2<23,20,np.maximum(0,20*(26-t2)/3)))
profile_b=np.where(t2<3,10*t2/3,np.where(t2<23,10,np.maximum(0,10*(26-t2)/3)))
sa=simulate_compartments(t2,profile_a,half_times)
sb=simulate_compartments(t2,profile_b,half_times)
print("At the common surface endpoint:")
for h,a_state,b_state in zip(half_times,sa[:,-1],sb[:,-1]):
    print(f"{h:4.0f} min compartment: history A {a_state:.3f}, history B {b_state:.3f} bar")
assert np.all(sa[:,-1]>sb[:,-1])

## 6. Euler error versus exact update

In [ ]:
def euler_constant(dt,half=5,duration=20):
    n=int(duration/dt); p=p_surface; pi=inspired_n2_bar(20); k=np.log(2)/half
    for _ in range(n): p += dt*k*(pi-p)
    return p

truth=constant_response(20,p_surface,inspired_n2_bar(20),5)
for dt in [1.0,0.25,0.05]:
    print(f"dt={dt:.2f} min, Euler error={euler_constant(dt)-truth:+.5f} bar")

## Engineering exercises

1. Add compartments with half-times 10, 40, and 160 minutes.
2. Compare uptake and elimination for the same compartment.
3. Create two profiles with identical maximum depth and duration but different final state vectors.
4. Quantify Euler error as a function of step size.
5. Explain what additional evidence would be required before any limit-setting rule could be trusted.

## Summary

Each compartment is a stable first-order state whose eigenvalue is fixed by its half-time. A bank of such states stores pressure history across many time scales, but state propagation alone is neither a physiological truth nor an operational decompression algorithm.